# XiDepth v2.0 — Track 1b: ResNet-18 From-Scratch Control Run

This notebook trains the ResNet-18 baseline **from scratch** (random initialization, `--pretrained` omitted),
providing a direct, like-for-like control group against the novel XiDepthNet architecture.

In [ ]:
# 1. Verify GPU & Environment
!nvidia-smi
import os, glob
input_dirs = glob.glob("/kaggle/input/*kitti*")
dataset_dir = input_dirs[0] if input_dirs else "/kaggle/input/kitti-eigen-split"
print(f"Using dataset directory: {dataset_dir}")


In [ ]:
# 2. Clone audited branch v2-eval-audit
!git clone -b v2-eval-audit https://github.com/ASMITSARKAR/XiDepth-Monocular-2.0.git xidepth_repo
%cd xidepth_repo
!pip install -r requirements.txt
!mkdir -p data
!wget -nc --show-progress "https://www.dropbox.com/scl/fi/dg7eskv5ztgdyp4ippqoa/gt_depths.npz?rlkey=qb39aajkbhmnod71rm32136ry&dl=1" -O data/gt_depths.npz


In [ ]:
# 3. Launch Track 1b Control Training (From Scratch, NO --pretrained flag)
!python scripts/train.py \
    --model resnet18 \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_resnet18_scratch \
    --batch_size 12 \
    --epochs 10 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100

# To resume in a subsequent session:
# !python scripts/train.py \
#     --model resnet18 \
#     --resume /kaggle/input/my-previous-scratch-run/checkpoint_epoch_10.pth \
#     --dataset_dir "$dataset_dir" \
#     --split_dir data/splits/eigen_zhou \
#     --checkpoint_dir /kaggle/working/checkpoints_resnet18_scratch \
#     --batch_size 12 \
#     --epochs 20 \
#     --num_workers 4 \
#     --use_amp


In [ ]:
# 4. Evaluate Track 1b From-Scratch Model on Eigen Benchmark (652 files)
!python scripts/eval.py \
    --model resnet18 \
    --checkpoint /kaggle/working/checkpoints_resnet18_scratch/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/resnet18_scratch_results.json
